<a href="https://colab.research.google.com/github/sarahshahrir/ligand-docking/blob/experiment/DiffDock_Exp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install Dependencies


In [1]:
# Install PyTorch with CUDA support first
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Verify GPU is working before continuing
import torch
print(torch.__version__)
print(torch.cuda.is_available())  # must be True before proceeding

# Install PyG and dependencies matching the CUDA version
!pip install torch-geometric
!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv \
  -f https://data.pyg.org/whl/torch-2.10.0+cu128.html

# Install remaining dependencies
!pip install e3nn biopython spyrmsd
!pip install pandas scipy scikit-learn pyyaml tqdm networkx
!pip install rdkit
!pip install fair-esm==2.0.0
!pip install prody
!pip install py3Dmol
!pip install biopandas

Looking in indexes: https://download.pytorch.org/whl/cu121
2.10.0+cu128
True
Looking in links: https://data.pyg.org/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 40.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 74.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 75.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 66.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 57.8 MB/s eta 0:00:00
  Using cached py3dmol-2.5.4-py2.py3-none-any.whl.metadata (2.1 kB)
Using cached py3dmol-2.5.4-py2.py3-none-any.whl (7.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 kB 3.5 MB/s eta 0:00:00


# Clone DiffDock

In [2]:
!git clone https://github.com/gcorso/DiffDock.git
%cd DiffDock

Cloning into 'DiffDock'...
remote: Enumerating objects: 520, done.
remote: Counting objects: 100% (293/293), done.
remote: Compressing objects: 100% (139/139), done.
remote: Total 520 (delta 209), reused 154 (delta 154), pack-reused 227 (from 3)
Receiving objects: 100% (520/520), 233.08 MiB | 37.34 MiB/s, done.
Resolving deltas: 100% (244/244), done.
/content/DiffDock


In [ ]:
# ATTENTION: Please replace the last two lines of /content/DiffDock/datasets/esm_embedding_preparation.py
# with the following code
"""
records = []
    for seq_id, seq in name_to_sequence.items():
        record = SeqRecord(Seq(seq), id=str(seq_id))
        record.description = ''
        records.append(record)
    SeqIO.write(records, args.out_file, "fasta")
"""
# Failing to replace may cause errors in inference process.

# Download Processed Datasets
PDBBind processed data has copyright infringement, currently unavailable.
Here, we use BindingMoad instead.

In [3]:
# Download
!wget -O BindingMOAD_2020_processed.tar \
  "https://zenodo.org/records/10656052/files/BindingMOAD_2020_processed.tar?download=1"

--2026-04-15 00:39:24--  https://zenodo.org/records/10656052/files/BindingMOAD_2020_processed.tar?download=1
Resolving zenodo.org (zenodo.org)... 188.185.48.75, 137.138.153.219, 188.185.43.153, ...
Connecting to zenodo.org (zenodo.org)|188.185.48.75|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 28543702016 (27G) [application/octet-stream]
Saving to: ‘BindingMOAD_2020_processed.tar’

BindingMOAD_2020_pr 100%[===================>]  26.58G  16.5MB/s    in 27m 44s 

2026-04-15 01:07:08 (16.4 MB/s) - ‘BindingMOAD_2020_processed.tar’ saved [28543702016/28543702016]



In [4]:
# Extract
!mkdir -p data/BindingMOAD_2020_processed
!tar -xf BindingMOAD_2020_processed.tar -C data/BindingMOAD_2020_processed

# Inspect the folder structure
!find data/BindingMOAD_2020_processed -maxdepth 2 -type d | sort | head -50

Streaming output truncated to the last 5000 lines.
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.quarantine'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.quarantine'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.quarantine'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.quarantine'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.quarantine'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.quarantine'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.quarantine'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.quarantine'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.quarantine'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.quarantine'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.quarantine'
tar

In [6]:
# Generate the ESM2 embeddings
!cd /content/DiffDock \
&& PYTHONPATH=/content/DiffDock python datasets/esm_embedding_preparation.py \
  --data_dir /content/DiffDock/data/BindingMOAD_2020_processed/BindingMOAD_2020_processed/pdb_protein \
  --dataset moad

  0% 0/3 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/Bio/PDB/PDBParser.py:384: PDBConstructionWarning: Ignoring unrecognized record 'TER' at line 1633
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/Bio/PDB/PDBParser.py:384: PDBConstructionWarning: Ignoring unrecognized record 'TER' at line 3265
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/Bio/PDB/PDBParser.py:384: PDBConstructionWarning: Ignoring unrecognized record 'TER' at line 943
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/Bio/PDB/PDBParser.py:384: PDBConstructionWarning: Ignoring unrecognized record 'TER' at line 1885
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/Bio/PDB/PDBParser.py:384: PDBConstructionWarning: Ignoring unrecognized record 'TER' at line 2561
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/Bio/PDB/PDBParser.py:384: PDBConstructionWarning: Ignoring unrecognized record 'TER' at line 5121
  warnings.warn(
100% 3/3 [00:00<00:00, 26.70it/s]


In [7]:
# Create a csv containing paths to protein and ligand
import os
import pandas as pd

root = "data/BindingMOAD_2020_processed/BindingMOAD_2020_processed"
protein_dir = os.path.join(root, "pdb_protein")
ligand_dir = os.path.join(root, "pdb_superligand")

rows = []

all_ligands = os.listdir(ligand_dir)

for protein_file in os.listdir(protein_dir):
    if protein_file.endswith("_protein.pdb") and not protein_file.startswith("._"):
        base = protein_file.replace("_protein.pdb", "")
        protein_path = os.path.join(protein_dir, protein_file)

        matches = sorted([
            f for f in all_ligands
            if f.startswith(f"{base}_superlig_")
            and f.endswith(".pdb")
            and not f.startswith("._")
        ])

        if matches:
            ligand_path = os.path.join(ligand_dir, matches[0])  # take first matching superligand
            rows.append({
                "protein_path": protein_path,
                "ligand": ligand_path
            })

df = pd.DataFrame(rows)
df.to_csv("data/moad_for_esm.csv", index=False)

print(df.head())
print("Number of rows:", len(df))

                                     protein_path  \
0  data/moad_small/pdb_protein/10gs_1_protein.pdb   
1  data/moad_small/pdb_protein/11ba_1_protein.pdb   
2  data/moad_small/pdb_protein/11as_1_protein.pdb   

                                              ligand  
0  data/moad_small/pdb_superligand/10gs_1_superli...  
1  data/moad_small/pdb_superligand/11ba_1_superli...  
2  data/moad_small/pdb_superligand/11as_1_superli...  
Number of rows: 3


# Generate ESM2 embeddings for Proteins
DiffDock -> FASTA -> ESM -> embeddings -> DiffDock

In [8]:
# This command extracts sequences from protein structures, puts them all in a FASTA file.
!cd /content/DiffDock && PYTHONPATH=/content/DiffDock python -m datasets.esm_embedding_preparation \
  --dataset moad \
  --data_dir /content/DiffDock/data/BindingMOAD_2020_processed/BindingMOAD_2020_processed/pdb_protein \
  --out_file data/moad_sequences.fasta

!head -5 /content/DiffDock/data/moad_sequences.fasta

  0% 0/3 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/Bio/PDB/PDBParser.py:384: PDBConstructionWarning: Ignoring unrecognized record 'TER' at line 1633
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/Bio/PDB/PDBParser.py:384: PDBConstructionWarning: Ignoring unrecognized record 'TER' at line 3265
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/Bio/PDB/PDBParser.py:384: PDBConstructionWarning: Ignoring unrecognized record 'TER' at line 943
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/Bio/PDB/PDBParser.py:384: PDBConstructionWarning: Ignoring unrecognized record 'TER' at line 1885
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/Bio/PDB/PDBParser.py:384: PDBConstructionWarning: Ignoring unrecognized record 'TER' at line 2561
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/Bio/PDB/PDBParser.py:384: PDBConstructionWarning: Ignoring unrecognized record 'TER' at line 5121
  warnings.warn(
100% 3/3 [00:00<00:00, 27.93it/s]
>10gs_1_chai

In [9]:
# Verify FASTA content
!file /content/DiffDock/data/moad_sequences.fasta
!head -5 /content/DiffDock/data/moad_sequences.fasta

/content/DiffDock/data/moad_sequences.fasta: ASCII text
>10gs_1_chain_0
PYTVVYFPVRGRCAALRMLLADQGQSWKEEVVTVETWQEGSLKASCLYGQLPKFQDGDLT
LYQSNTILRHLGRTLGLYGKDQQEAALVDMVNDGVEDLRCKYISLIYTNYEAGKDDYVKA
LPGQLKPFETLLSQNQGGKTFIVGDQISFADYNLLDLLLIHEVLAPGCLDAFPLLSAYVG
RLSARPKLKAFLASPEYVNLPINGNGKQ


In [10]:
# Run esm to get embedding
!git clone https://github.com/facebookresearch/esm.git /content/esm

!pip install -e /content/esm

!python /content/esm/scripts/extract.py \
  esm2_t33_650M_UR50D \
  /content/DiffDock/data/moad_sequences.fasta \
  /content/DiffDock/data/embeddings_output \
  --repr_layers 33 \
  --include per_tok \
  --truncation_seq_length 4096

Cloning into '/content/esm'...
remote: Enumerating objects: 1511, done.
remote: Counting objects: 100% (807/807), done.
remote: Compressing objects: 100% (215/215), done.
remote: Total 1511 (delta 630), reused 592 (delta 592), pack-reused 704 (from 1)
Receiving objects: 100% (1511/1511), 12.73 MiB | 27.56 MiB/s, done.
Resolving deltas: 100% (953/953), done.
Obtaining file:///content/esm
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for fair-esm (pyproject.toml) ... done
  Created wheel for fair-esm: filename=fair_esm-2.0.1-0.editable-py3-none-any.whl size=18077 sha256=661309d7760100cc5f70b89ab47c20ec5df3ffd365dbb346d51df2138887640b
  Stored in directory: /tmp/pip-ephem-wheel-cache-lx4hrc0_/wheels/c8/40/16/eb6ca5c9e531433ed45ec048c279dbfd220338e09a1dcb33fc
Successfully built fair-esm
  Attempting uninstall: 

In [11]:
# Move embeddings folder to data folder
!mv /content/DiffDock/data/embeddings_output /content/DiffDock/data/
%cd ..

mv: '/content/DiffDock/data/embeddings_output' and '/content/DiffDock/data/embeddings_output' are the same file
/content


In [12]:
# Merge all pt file in embeddings folder into one
import torch
import os

embeddings_dir = "/content/DiffDock/data/embeddings_output"
merged = {}

for fname in os.listdir(embeddings_dir):
    if fname.endswith(".pt"):
        key = fname.replace(".pt", "")  # e.g. "10gs_1_chain_0"
        data = torch.load(os.path.join(embeddings_dir, fname))
        # ESM stores embeddings under 'representations', grab layer 33
        merged[key] = data["representations"][33]

torch.save(merged, "/content/DiffDock/data/moad_esm2_embeddings.pt")
print(f"Merged {len(merged)} embeddings into moad_esm2_embeddings.pt")

Merged 6 embeddings into moad_esm2_embeddings.pt


In [13]:
# Verify merged esm2 embeddings pt
emb = torch.load("/content/DiffDock/data/moad_esm2_embeddings.pt")
print(list(emb.keys())[:5])        # should show chain names like '10gs_1_chain_0'
print(list(emb.values())[0].shape) # should be (seq_len, 1280)

['10gs_1_chain_0', '11ba_1_chain_1', '11ba_1_chain_0', '11as_1_chain_0', '10gs_1_chain_1']
torch.Size([208, 1280])


## Run DiffDock

In [14]:
# Put paths to protein and ligand files in one csv.
# Small pipeline, CHANGE SMALL_MOAD
import pandas as pd
import os
import glob

protein_dir = '/content/DiffDock/data/BindingMOAD_2020_processed/BindingMOAD_2020_processed/pdb_protein'
ligand_dir = '/content/DiffDock/data/BindingMOAD_2020_processed/BindingMOAD_2020_processed/pdb_superligand'

protein_files = glob.glob(os.path.join(protein_dir, '*.pdb'))
print(f"Found {len(protein_files)} protein files")

rows = []
skipped = []

for protein_path in protein_files:
    basename = os.path.basename(protein_path)
    complex_name = basename.replace('_protein.pdb', '')

    # Match any superligand index (0, 1, 2...) for this complex
    ligand_pattern = os.path.join(ligand_dir, f'{complex_name}_superlig_*.pdb')
    ligand_files = sorted(glob.glob(ligand_pattern))

    if ligand_files:
        # One row per ligand
        for ligand_path in ligand_files:
            lig_basename = os.path.basename(ligand_path)
            lig_idx = lig_basename.replace(f'{complex_name}_superlig_', '').replace('.pdb', '')
            rows.append({
                'complex_name': f'{complex_name}_lig{lig_idx}',
                'protein_path': f'data/BindingMOAD_2020_processed/BindingMOAD_2020_processed/pdb_protein/{basename}',
                'ligand_description': f'data/BindingMOAD_2020_processed/BindingMOAD_2020_processed/pdb_superligand/{lig_basename}',
                'protein_sequence': ''
            })
    else:
        skipped.append(complex_name)

df = pd.DataFrame(rows)
df.to_csv('/content/DiffDock/data/bindingmoad_full.csv', index=False)

print(f"Created CSV with {len(df)} complexes")
print(f"Skipped {len(skipped)} proteins with no ligand: {skipped[:5]}...")
print(df.head())

Found 3 protein files
Created CSV with 3 complexes
Skipped 0 proteins with no ligand: []...
  complex_name                                    protein_path  \
0  10gs_1_lig0  data/moad_small/pdb_protein/10gs_1_protein.pdb   
1  11ba_1_lig0  data/moad_small/pdb_protein/11ba_1_protein.pdb   
2  11as_1_lig0  data/moad_small/pdb_protein/11as_1_protein.pdb   

                                  ligand_description protein_sequence  
0  data/moad_small/pdb_superligand/10gs_1_superli...                   
1  data/moad_small/pdb_superligand/11ba_1_superli...                   
2  data/moad_small/pdb_superligand/11as_1_superli...                   


In [15]:
# Small sample, CHANGE DATASET DIR
%cd /content/DiffDock

!python -m inference \
  --config default_inference_args.yaml  \
  --protein_ligand_csv data/bindingmoad_full.csv \
  --out_dir results/user_predictions_small

/content/DiffDock
/content/DiffDock/utils/so3.py:59: RuntimeWarning: invalid value encountered in sqrt
  _exp_score_norms = np.sqrt(np.sum(_score_norms**2 * _pdf_vals, axis=1) / np.sum(_pdf_vals, axis=1) / np.pi)
100% 201/201 [01:00<00:00,  3.30it/s]
100% 201/201 [01:21<00:00,  2.46it/s]
Generating ESM language model embeddings
Processing 1 of 1 batches (6 sequences)
0it [00:00, ?it/s]/content/DiffDock/datasets/parse_chi.py:91: RuntimeWarning: invalid value encountered in cast
  Y = indices.astype(int)
3it [00:30, 10.06s/it]
